<a href="https://colab.research.google.com/github/Claudiopj88/Atividades_ANN/blob/main/Atividade01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dataset

In [1]:
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split

X, y = load_digits(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Funções de Ativação

In [2]:
import numpy as np
from scipy.special import softmax
from abc import ABC, abstractmethod

class Activation(ABC):
    @abstractmethod
    def forward(self, z):
        pass

    @abstractmethod
    def backward(self, z):
        pass

class Tanh(Activation):
    def forward(self, z):
        return np.tanh(z)

    def backward(self, z):
        return 1 - np.tanh(z)**2

class Sigmoid(Activation):
    def forward(self, z):
        return 1 / (1 + np.exp(-z))

    def backward(self, z):
        s = self.forward(z)
        return s * (1 - s)

class ReLU(Activation):
    def forward(self, z):
        return np.maximum(0, z)

    def backward(self, z):
        return (z > 0).astype(float)

class Softmax(Activation):
    def forward(self, z):
        exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
        return exp_z / np.sum(exp_z, axis=1, keepdims=True)

    def backward(self, z):
        pass

def get_activation(name: str) -> Activation:
    match name.lower():
        case 'tanh':
            return Tanh()
        case 'sigmoid':
            return Sigmoid()
        case 'relu':
            return ReLU()
        case 'softmax':
            return Softmax()





# Rede Neural

In [3]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import LabelBinarizer

def include_bias(X):
  return np.hstack((np.ones((X.shape[0], 1)), X))

def sign(a):
  return (a>=0)*2-1

class MultiLayer(BaseEstimator, ClassifierMixin):
  def __init__(self, max_iter=2000, learning_rate=0.001,
               n_hidden=[100], activation='tanh'):
    self.max_iter = max_iter
    self.learning_rate = learning_rate
    self.n_hidden = n_hidden
    self.act_layer = get_activation(activation)
    self.output_layer = Softmax()


  def forward(self, X):
    self.A = []
    self.Z = []
    AUX = X.copy()
    for i in range(len(self.Ws) -1 ):
      self.A.append(include_bias(AUX))
      self.Z.append(self.A[-1] @ self.Ws[i])
      AUX = self.act_layer.forward(self.Z[-1])

    # Última camada forward
    self.A.append(include_bias(AUX))
    self.Z.append(self.A[-1] @ self.Ws[-1])
    y_prob = self.output_layer.forward(self.Z[-1])
    return y_prob

  def backward(self, y, y_prob):
    grads = []
    output_delta = y_prob - y
    grads.insert(0, self.A[-1].T @ output_delta)

    for i in range(len(self.Ws)-1, 0, -1):
      func_grad = self.act_layer.backward(self.Z[i-1])
      input_delta = output_delta @ self.Ws[i][1:].T * func_grad
      grads.insert(0, self.A[i-1].T @ input_delta)
      output_delta = input_delta.copy()
    for i in range(len(self.Ws)):
      self.Ws[i] -= grads[i] * self.learning_rate

  def fit(self, X, y):
    self.lb = LabelBinarizer()
    y_encoded = self.lb.fit_transform(y)
    if len(y_encoded.shape) == 1:
      y_encoded = np.hstack((1 - y_encoded, y_encoded))

    self.Ws = []
    previous_output = X.shape[1]
    for n in self.n_hidden:
      self.Ws.append(np.random.uniform(-1, 1, (previous_output+1, n)))
      previous_output = n
    self.Ws.append(np.random.uniform(-1, 1, (previous_output+1, y_encoded.shape[1])))
    for _ in range(self.max_iter):
      y_prob = self.forward(X)
      self.backward(y_encoded, y_prob)
    return self

  def predict(self, X):
    y_prob = self.forward(X)
    return self.lb.inverse_transform(y_prob)


# Treinamento

In [9]:
model = MultiLayer(activation='tanh', n_hidden=[100], max_iter=2000, learning_rate=0.001)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9388888888888889


In [10]:
model = MultiLayer(activation='sigmoid',  n_hidden=[100], max_iter=2000, learning_rate=0.001 )
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9611111111111111


In [12]:
model = MultiLayer(activation='relu',n_hidden=[100], max_iter=2000, learning_rate=0.0001)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9388888888888889


In [13]:
from sklearn.neural_network import MLPClassifier

model = MLPClassifier(max_iter=2000)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)
print(accuracy_score(y_test, y_pred))

0.9722222222222222
